# Pollution as the endogenous variable (MERRA-2)

The main specification instruments the district's own fire intensity (log FRP) with upwind fire. Because smoke from upwind fires could affect conflict directly, the exclusion restriction for that IV is hard to defend, and the paper treats the reduced form as primary. This notebook replaces fire intensity with a direct measure of **air pollution**, so the IV estimates the effect of pollution itself:

* **Surface PM2.5** (µg/m³) from MERRA-2 monthly aerosol diagnostics (M2TMNXAER), built from the standard formula: dust (< 2.5 µm) + sea salt (< 2.5 µm) + organic carbon + black carbon + 1.375 × sulfate (Buchard et al. 2016).
* **Aerosol optical depth** (AOD at 550 nm, `TOTEXTTAU`) as a second measure.

The exclusion restriction becomes: upwind fire affects conflict only through the pollution it brings. **Caveat:** MERRA-2 is a reanalysis. It assimilates satellite AOD, but its biomass-burning emissions come from QFED, which is built from MODIS fire detections, so its PM2.5 is partly derived from fire data of the kind the instrument uses.

Downloads use NASA Earthdata credentials in `~/.netrc` (no credentials are stored in this notebook). Each global monthly file (~84 MB) is downloaded, cropped to Indonesia and deleted; only the cropped file is kept.

In [1]:
import os
import re
import warnings
import subprocess
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import pyfixest as pf
from scipy import stats
from scipy.spatial import cKDTree

warnings.filterwarnings('ignore')

ROOT         = '/Users/hariaksha/Documents/GitHub/climate-conflict'
PANEL_CSV    = f'{ROOT}/analysis/iv_panel_district.csv'
DISTRICT_SHP = f'{ROOT}/data/administrative/gadm41_IDN_shp/gadm41_IDN_2.shp'
ACLED_XLSX   = f'{ROOT}/data/unrest/ACLED Data_2026-06-10.xlsx'
POLL_DIR     = f'{ROOT}/data/pollution'
RAW_DIR      = f'{POLL_DIR}/merra2_raw_tmp'
POLL_NC      = f'{POLL_DIR}/merra2_aer_indonesia_2015_2025.nc'
POLL_CSV     = f'{POLL_DIR}/merra2_pm25_district_monthly.csv'
OUT_CSV      = f'{ROOT}/analysis/output/pollution_iv_results.csv'

BASE_URL = 'https://goldsmr4.gesdisc.eosdis.nasa.gov/data/MERRA2_MONTHLY/M2TMNXAER.5.12.4'
VARS     = ['TOTEXTTAU', 'DUSMASS25', 'SSSMASS25', 'OCSMASS', 'BCSMASS', 'SO4SMASS']
LAT, LON = slice(-11, 6), slice(95, 141)
START_YEAR, END_YEAR, END_MONTH = 2015, 2025, 6

INSTR    = ['log_upwind_frp', 'log_upwind_frp_l1']
OUTCOMES = ['events', 'pv_events', 'riots_vac', 'vac']
FE       = 'district + year_month'
VCOV     = {'CRV1': 'province'}
os.makedirs(RAW_DIR, exist_ok=True)

## 1. Download and crop MERRA-2

File names depend on the MERRA-2 production stream (e.g. `MERRA2_400`, `MERRA2_401`), so each year's directory listing is read to find them. Skipped if the cropped file already exists.

In [2]:
def curl(args):
    # -n reads ~/.netrc; the cookie jar holds the Earthdata session across redirects
    cookie = f'{RAW_DIR}/.cookies'
    return subprocess.run(['curl', '-s', '-n', '-L', '-c', cookie, '-b', cookie] + args,
                          capture_output=True, check=True)

if not os.path.exists(POLL_NC):
    parts = []
    for year in range(START_YEAR, END_YEAR + 1):
        listing = curl([f'{BASE_URL}/{year}/']).stdout.decode()
        files = sorted(set(re.findall(r'MERRA2_\d+\.tavgM_2d_aer_Nx\.\d{6}\.nc4(?!\.)', listing)))
        for fn in files:
            ym = int(fn.split('.')[-2])
            if year == END_YEAR and ym % 100 > END_MONTH:
                continue
            path = f'{RAW_DIR}/{fn}'
            curl(['-o', path, f'{BASE_URL}/{year}/{fn}'])
            with xr.open_dataset(path) as ds:
                parts.append(ds[VARS].sel(lat=LAT, lon=LON).load())
            os.remove(path)
        print(f'{year}: {len(files)} files')
    xr.concat(parts, dim='time').to_netcdf(POLL_NC)
    print(f'Saved {len(parts)} months to {POLL_NC}')
else:
    print(f'{POLL_NC} already exists -- skipping download.')

ds = xr.open_dataset(POLL_NC)
print(ds)

/Users/hariaksha/Documents/GitHub/climate-conflict/data/pollution/merra2_aer_indonesia_2015_2025.nc already exists -- skipping download.


<xarray.Dataset> Size: 8MB
Dimensions:    (time: 126, lat: 35, lon: 74)
Coordinates:
  * time       (time) datetime64[ns] 1kB 2015-01-01T00:30:00 ... 2025-06-01T0...
  * lat        (lat) float64 280B -11.0 -10.5 -10.0 -9.5 ... 4.5 5.0 5.5 6.0
  * lon        (lon) float64 592B 95.0 95.62 96.25 96.88 ... 139.4 140.0 140.6
Data variables:
    TOTEXTTAU  (time, lat, lon) float32 1MB ...
    DUSMASS25  (time, lat, lon) float32 1MB ...
    SSSMASS25  (time, lat, lon) float32 1MB ...
    OCSMASS    (time, lat, lon) float32 1MB ...
    BCSMASS    (time, lat, lon) float32 1MB ...
    SO4SMASS   (time, lat, lon) float32 1MB ...
Attributes: (12/30)
    History:                           Original file generated: Mon Mar 23 19...
    Filename:                          MERRA2_400.tavgM_2d_aer_Nx.201501.nc4
    Comment:                           GMAO filename: d5124_m2_jan10.tavg1_2d...
    Conventions:                       CF-1
    Institution:                       NASA Global Modeling and Assimil

## 2. Aggregate to district-months

The MERRA-2 grid is coarse (0.5° × 0.625°, roughly 55 × 70 km), so many districts contain no grid-cell centre. Each district is assigned the mean of the cells whose centres fall inside it; districts containing none use the cell nearest their representative point (as for ERA5 in `weather-controls.ipynb`). Mass concentrations are converted from kg/m³ to µg/m³.

In [3]:
districts_gdf = gpd.read_file(DISTRICT_SHP)[['NAME_1', 'NAME_2', 'geometry']].rename(
    columns={'NAME_1': 'province', 'NAME_2': 'district'})
dup = districts_gdf['district'].duplicated(keep=False)
districts_gdf.loc[dup, 'district'] = (districts_gdf.loc[dup, 'district'] + ' ('
                                      + districts_gdf.loc[dup, 'province'] + ')')
rep = districts_gdf.geometry.representative_point()
districts_gdf['dist_lat'], districts_gdf['dist_lon'] = rep.y, rep.x

px = ds.to_dataframe().reset_index()
px['time'] = pd.to_datetime(px['time'])
px['year'], px['month'] = px['time'].dt.year, px['time'].dt.month
px['pm25'] = (px['DUSMASS25'] + px['SSSMASS25'] + px['OCSMASS'] + px['BCSMASS']
              + px['SO4SMASS'] * 132.14 / 96.06) * 1e9
px['aod'] = px['TOTEXTTAU']

grid = px[['lat', 'lon']].drop_duplicates().reset_index(drop=True)
gg = gpd.GeoDataFrame(grid, geometry=gpd.points_from_xy(grid['lon'], grid['lat']), crs='EPSG:4326')
joined = gpd.sjoin(gg, districts_gdf[['district', 'geometry']], how='left', predicate='within')
joined = joined[~joined.index.duplicated(keep='first')]
grid['district'] = joined['district'].values
missing = districts_gdf[~districts_gdf['district'].isin(set(grid['district'].dropna()))]
_, nn = cKDTree(grid[['lat', 'lon']].values).query(missing[['dist_lat', 'dist_lon']].values)
fallback = grid.iloc[nn][['lat', 'lon']].reset_index(drop=True)
fallback['district'] = missing['district'].values
grid = pd.concat([grid.dropna(subset=['district']), fallback], ignore_index=True)
print(f'Districts with pollution data: {grid["district"].nunique()} ({len(missing)} via nearest-cell fallback)')

poll = (px.merge(grid, on=['lat', 'lon'])
          .groupby(['district', 'year', 'month'])[['pm25', 'aod']].mean().reset_index())
poll.to_csv(POLL_CSV, index=False)
print(f'Saved: {POLL_CSV}')
print(poll[['pm25', 'aod']].describe().round(2))
print('\nMean PM2.5 by month (haze season is Aug-Oct):')
print(poll.groupby('month')['pm25'].mean().round(1).to_dict())

Districts with pollution data: 502 (243 via nearest-cell fallback)


Saved: /Users/hariaksha/Documents/GitHub/climate-conflict/data/pollution/merra2_pm25_district_monthly.csv
           pm25       aod
count  63252.00  63252.00
mean      11.36      0.17
std        7.56      0.12
min        1.04      0.02
25%        6.91      0.10
50%        9.64      0.14
75%       14.24      0.22
max      239.07      1.85

Mean PM2.5 by month (haze season is Aug-Oct):
{1: 9.300000190734863, 2: 10.399999618530273, 3: 10.300000190734863, 4: 10.5, 5: 10.800000190734863, 6: 10.899999618530273, 7: 11.5, 8: 12.399999618530273, 9: 15.199999809265137, 10: 15.199999809265137, 11: 11.100000381469727, 12: 9.300000190734863}


## 3. Merge into the IV panel

Pollution enters in logs, with a one-month lag, matching the fire variables.

In [4]:
panel = pd.read_csv(PANEL_CSV)
poll = poll.sort_values(['district', 'year', 'month'])
for v in ['pm25', 'aod']:
    poll[f'log_{v}'] = np.log(poll[v])
    poll[f'log_{v}_l1'] = poll.groupby('district')[f'log_{v}'].shift(1)
panel = panel.merge(poll, on=['district', 'year', 'month'], how='left')
panel = panel.dropna(subset=['log_pm25', 'log_pm25_l1']).reset_index(drop=True)
print(f'Panel: {len(panel):,} obs, {panel["district"].nunique()} districts')
print(f'Corr(log upwind FRP, log PM2.5): {panel["log_upwind_frp"].corr(panel["log_pm25"]):.3f}')
print(f'Corr(log own FRP,    log PM2.5): {panel["log_frp"].corr(panel["log_pm25"]):.3f}')

Panel: 60,125 obs, 481 districts
Corr(log upwind FRP, log PM2.5): 0.249
Corr(log own FRP,    log PM2.5): 0.076


## 4. Event-type outcomes

As in `weather-controls.ipynb`, Section 4.

In [5]:
districts_gdf = gpd.read_file(DISTRICT_SHP)[['NAME_1', 'NAME_2', 'geometry']].rename(
    columns={'NAME_1': 'province', 'NAME_2': 'district'})
dup = districts_gdf['district'].duplicated(keep=False)
districts_gdf.loc[dup, 'district'] = (districts_gdf.loc[dup, 'district'] + ' ('
                                      + districts_gdf.loc[dup, 'province'] + ')')

acled = pd.read_excel(ACLED_XLSX, usecols=['event_date', 'event_type', 'latitude', 'longitude'])
acled['event_date'] = pd.to_datetime(acled['event_date'])
acled['year'], acled['month'] = acled['event_date'].dt.year, acled['event_date'].dt.month
ag = gpd.GeoDataFrame(acled, geometry=gpd.points_from_xy(acled['longitude'], acled['latitude']), crs='EPSG:4326')
j = gpd.sjoin(ag, districts_gdf[['district', 'geometry']], how='left', predicate='within')
j = j[~j.index.duplicated(keep='first')]
acled['district'] = j['district'].reindex(acled.index).values
off = acled['district'].isna()
near = gpd.sjoin_nearest(ag.loc[off].to_crs(3395), districts_gdf[['district', 'geometry']].to_crs(3395),
                         how='left', max_distance=20_000)
near = near[~near.index.duplicated(keep='first')]
acled.loc[off, 'district'] = near['district'].reindex(acled.index[off]).values

acled['riots_vac'] = acled['event_type'].isin(['Riots', 'Violence against civilians']).astype(int)
acled['vac'] = (acled['event_type'] == 'Violence against civilians').astype(int)
counts = acled.groupby(['district', 'year', 'month'])[['riots_vac', 'vac']].sum().reset_index()
panel = panel.merge(counts, on=['district', 'year', 'month'], how='left').fillna({'riots_vac': 0, 'vac': 0})
print(panel[OUTCOMES].sum())

events       21653.0
pv_events     3520.0
riots_vac     2824.0
vac           1067.0
dtype: float64


## 5. Estimates

For each pollution measure and sample: the first stage (log pollution on contemporaneous and lagged log upwind FRP, with lagged log pollution as a control), the effective F, the 2SLS coefficient on instrumented log pollution, and the Anderson–Rubin p-value. A second version also controls for the district's own fire (log FRP and its lag), so the pollution coefficient is not picking up a direct effect of local burning. Samples: full panel, conflict-active (≥ 30% of months), and conflict-active restricted to 2018–2025.

In [6]:
def effective_f(data, instruments, controls=(), fe=FE, endog='log_frp'):
    data = data.reset_index(drop=True)
    ctrl = ' + '.join(controls)
    rhs = ' + '.join(list(instruments) + list(controls))
    fs = pf.feols(f'{endog} ~ {rhs} | {fe}', data=data, vcov=VCOV)
    Z = np.column_stack([pf.feols(f'{z} ~ {ctrl if ctrl else "1"} | {fe}', data=data).resid()
                         for z in instruments])
    names = list(fs.coef().index)
    idx = [names.index(z) for z in instruments]
    pi = fs.coef()[list(instruments)].to_numpy()
    V = fs._vcov[np.ix_(idx, idx)]
    S = Z.T @ Z
    return float(pi @ S @ pi / np.trace(V @ S))

def wald_p(m, terms, G):
    names = list(m.coef().index)
    idx = [names.index(t) for t in terms]
    b = m.coef()[terms].to_numpy()
    V = m._vcov[np.ix_(idx, idx)]
    W = float(b @ np.linalg.solve(V, b))
    return stats.f.sf(W / len(terms), len(terms), G - 1)

def ar_p(data, y, controls, fe=FE):
    m = pf.feols(f'{y} ~ {" + ".join(INSTR + list(controls))} | {fe}', data=data, vcov=VCOV)
    return wald_p(m, INSTR, data['province'].nunique())


share = panel.groupby('district')['events'].apply(lambda s: (s > 0).mean())
active = share[share >= 0.30].index
samples = {
    'full panel': panel,
    'active >= 30%': panel[panel['district'].isin(active)],
    'active >= 30%, 2018+': panel[panel['district'].isin(active) & (panel['year'] >= 2018)],
}
print(f'Active >= 30% districts: {len(active)}')

rows = []
for endog in ['log_pm25', 'log_aod']:
    for ctrl_label, extra in [('baseline', []), ('+ own fire', ['log_frp', 'log_frp_l1'])]:
        ctrls = [f'{endog}_l1'] + extra
        for sample, d in samples.items():
            d = d.dropna(subset=ctrls).reset_index(drop=True)
            effF = effective_f(d, INSTR, ctrls, endog=endog)
            fs = pf.feols(f'{endog} ~ {" + ".join(INSTR + ctrls)} | {FE}', data=d, vcov=VCOV).tidy()
            print(f'\n== {endog} | {ctrl_label} | {sample} | effective F = {effF:.1f} | '
                  f'first stage {fs.loc["log_upwind_frp", "Estimate"]:+.4f} (p={fs.loc["log_upwind_frp", "Pr(>|t|)"]:.3f}) ==')
            for y in OUTCOMES:
                iv = pf.feols(f'{y} ~ {" + ".join(ctrls)} | {FE} | {endog} ~ {" + ".join(INSTR)}',
                              data=d, vcov=VCOV).tidy().loc[endog]
                arp = ar_p(d, y, ctrls)
                rows.append({'endog': endog, 'controls': ctrl_label, 'sample': sample, 'outcome': y,
                             'effective_F': effF, 'first_stage': fs.loc['log_upwind_frp', 'Estimate'],
                             'iv_coef': iv['Estimate'], 'iv_se': iv['Std. Error'], 'iv_p': iv['Pr(>|t|)'],
                             'ar_p': arp, 'mean_y': d[y].mean()})
                print(f"  {y:10s} IV {iv['Estimate']:+.4f} (se {iv['Std. Error']:.4f}, p={iv['Pr(>|t|)']:.3f}) | AR p={arp:.3f}")

res = pd.DataFrame(rows)
res.to_csv(OUT_CSV, index=False)
print(f'\nSaved: {OUT_CSV}')

Active >= 30% districts: 85



== log_pm25 | baseline | full panel | effective F = 41.1 | first stage +0.0275 (p=0.000) ==


  events     IV -0.0792 (se 0.0760, p=0.304) | AR p=0.074


  pv_events  IV -0.0229 (se 0.0207, p=0.277) | AR p=0.021


  riots_vac  IV -0.0255 (se 0.0214, p=0.242) | AR p=0.027


  vac        IV -0.0232 (se 0.0156, p=0.147) | AR p=0.058

== log_pm25 | baseline | active >= 30% | effective F = 25.7 | first stage +0.0212 (p=0.000) ==
  events     IV -1.2271 (se 0.4056, p=0.005) | AR p=0.004
  pv_events  IV -0.3391 (se 0.1751, p=0.062) | AR p=0.154


... (19 outputs left out)

  riots_vac  IV -0.0205 (se 0.0403, p=0.614) | AR p=0.021


  vac        IV -0.0324 (se 0.0293, p=0.276) | AR p=0.083

== log_aod | + own fire | active >= 30% | effective F = 14.3 | first stage +0.0147 (p=0.000) ==
  events     IV -1.6575 (se 0.8428, p=0.058) | AR p=0.033
  pv_events  IV -0.4443 (se 0.3672, p=0.235) | AR p=0.160


  riots_vac  IV -0.4513 (se 0.3305, p=0.182) | AR p=0.077
  vac        IV -0.3148 (se 0.2472, p=0.212) | AR p=0.027

== log_aod | + own fire | active >= 30%, 2018+ | effective F = 10.1 | first stage +0.0147 (p=0.000) ==
  events     IV -2.1058 (se 1.0796, p=0.060) | AR p=0.038


  pv_events  IV -0.6810 (se 0.5775, p=0.247) | AR p=0.077
  riots_vac  IV -0.5691 (se 0.4463, p=0.211) | AR p=0.040
  vac        IV -0.4066 (se 0.3339, p=0.232) | AR p=0.009

Saved: /Users/hariaksha/Documents/GitHub/climate-conflict/analysis/output/pollution_iv_results.csv


## 6. Interpretation of magnitudes

The IV coefficient is the change in monthly events per one-unit change in log PM2.5, i.e. per roughly 2.7-fold increase. Below, it is rescaled to a 10% increase in PM2.5 and expressed relative to the sample mean of each outcome (baseline controls).

In [7]:
s = res[(res['endog'] == 'log_pm25') & (res['controls'] == 'baseline')].copy()
s['effect_10pct'] = s['iv_coef'] * np.log(1.1)
s['effect_10pct_rel_mean'] = s['effect_10pct'] / s['mean_y']
print(s[['sample', 'outcome', 'effective_F', 'iv_coef', 'iv_p', 'ar_p', 'mean_y',
         'effect_10pct', 'effect_10pct_rel_mean']].round(4).to_string(index=False))

              sample   outcome  effective_F  iv_coef   iv_p   ar_p  mean_y  effect_10pct  effect_10pct_rel_mean
          full panel    events      41.0728  -0.0792 0.3044 0.0735  0.3601       -0.0076                -0.0210
          full panel pv_events      41.0728  -0.0229 0.2766 0.0207  0.0585       -0.0022                -0.0373
          full panel riots_vac      41.0728  -0.0255 0.2424 0.0268  0.0470       -0.0024                -0.0518
          full panel       vac      41.0728  -0.0232 0.1472 0.0584  0.0177       -0.0022                -0.1244
       active >= 30%    events      25.6856  -1.2271 0.0049 0.0037  1.3975       -0.1170                -0.0837
       active >= 30% pv_events      25.6856  -0.3391 0.0618 0.1535  0.2176       -0.0323                -0.1485
       active >= 30% riots_vac      25.6856  -0.3723 0.0317 0.0863  0.1666       -0.0355                -0.2130
       active >= 30%       vac      25.6856  -0.2575 0.0507 0.0361  0.0719       -0.0245                